In [32]:
from pyspark.sql import SparkSession
import time

In [33]:
spark = SparkSession.builder \
    .appName("Apex Financial Data Ingestion") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

In [34]:
path = "c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/"
fact_transactions = spark.read.parquet(f"{path}fact_transactions.parquet")

fact_transactions.show(5)

+--------------+-----------+-----------+-----------+---------+-------------------+------+------------+----------+------------+---------------+---------------+------------+--------+
|transaction_id|customer_id|    card_id|merchant_id|device_id|          timestamp|amount|product_type| card_type|payment_type|customer_region|merchant_region|has_identity|is_fraud|
+--------------+-----------+-----------+-----------+---------+-------------------+------+------------+----------+------------+---------------+---------------+------------+--------+
|     T00000001|    C001131|CARD0001498|    M000467| D0001467|2026-05-26 21:43:58| 21.09|           W|      visa|       debit|           NULL|         Kisumu|       false|       0|
|     T00000002|    C001769|CARD0002353|    M000237| D0002277|2026-05-29 08:35:28| 36.01|           W|      visa|      credit|           NULL|         Kisumu|       false|       0|
|     T00000003|    C001770|CARD0002356|    M000752| D0002278|2026-04-07 14:35:18|158.71|      

In [35]:
transactions_100k = (
    fact_transactions
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
)

In [36]:
print("the count of transactions_100k is: ", transactions_100k.count())
print("the count of fact_transactions is: ", fact_transactions.count())


from pyspark.sql import functions as F
import time

rounds = [1,2,3]
partitions = [4, 6, 8, 10, 12, 14, 16]

for i in rounds:
    print(f"Round {i}:")
    for p in partitions:
        spark.conf.set("spark.sql.shuffle.partitions", str(p))

        start = time.time()

        result = (
            transactions_100k
            .groupBy("customer_id")
            .agg(
                F.count("transaction_id").alias("transaction_count"),
                F.sum("amount").alias("total_amount"),
                F.avg("amount").alias("avg_transaction_amount")
            )
        )

        result.count()  # action → actually executes the Spark job

        elapsed = time.time() - start

        print(f"{p} partitions: {elapsed:.3f} seconds")

the count of transactions_100k is:  100000
the count of fact_transactions is:  10000
Round 1:
4 partitions: 0.394 seconds
6 partitions: 0.346 seconds
8 partitions: 0.585 seconds
10 partitions: 0.842 seconds
12 partitions: 1.052 seconds
14 partitions: 0.821 seconds
16 partitions: 0.768 seconds
Round 2:
4 partitions: 0.711 seconds
6 partitions: 0.623 seconds
8 partitions: 0.996 seconds
10 partitions: 0.629 seconds
12 partitions: 0.630 seconds
14 partitions: 0.661 seconds
16 partitions: 0.831 seconds
Round 3:
4 partitions: 0.638 seconds
6 partitions: 0.780 seconds
8 partitions: 0.561 seconds
10 partitions: 0.708 seconds
12 partitions: 0.693 seconds
14 partitions: 0.710 seconds
16 partitions: 0.750 seconds


In [37]:
transactions_100k.rdd.getNumPartitions()

10

In [38]:
repartitioned = transactions_100k.repartition(20)

print("Original:", transactions_100k.rdd.getNumPartitions())
print("Repartitioned:", repartitioned.rdd.getNumPartitions())

Original: 10
Repartitioned: 20


In [39]:
repartitioned.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- == Final Plan ==
   ResultQueryStage (15)
   +- ShuffleQueryStage (14), Statistics(sizeInBytes=20.1 MiB, rowCount=1.00E+5)
      +- Exchange (13)
         +- * ColumnarToRow (12)
            +- Union (11)
               :- Scan parquet  (1)
               :- Scan parquet  (2)
               :- Scan parquet  (3)
               :- Scan parquet  (4)
               :- Scan parquet  (5)
               :- Scan parquet  (6)
               :- Scan parquet  (7)
               :- Scan parquet  (8)
               :- Scan parquet  (9)
               +- Scan parquet  (10)
+- == Initial Plan ==
   Exchange (16)
   +- Union (11)
      :- Scan parquet  (1)
      :- Scan parquet  (2)
      :- Scan parquet  (3)
      :- Scan parquet  (4)
      :- Scan parquet  (5)
      :- Scan parquet  (6)
      :- Scan parquet  (7)
      :- Scan parquet  (8)
      :- Scan parquet  (9)
      +- Scan parquet  (10)


(1) Scan parquet 
Output [14]: [transaction_id#2419, custom

In [40]:
coalesced = transactions_100k.coalesce(5)

print("Original:", transactions_100k.rdd.getNumPartitions())
print("Coalesced:", coalesced.rdd.getNumPartitions())

Original: 10
Coalesced: 5


In [41]:
coalesced.explain("formatted")

== Physical Plan ==
Coalesce (13)
+- * ColumnarToRow (12)
   +- Union (11)
      :- Scan parquet  (1)
      :- Scan parquet  (2)
      :- Scan parquet  (3)
      :- Scan parquet  (4)
      :- Scan parquet  (5)
      :- Scan parquet  (6)
      :- Scan parquet  (7)
      :- Scan parquet  (8)
      :- Scan parquet  (9)
      +- Scan parquet  (10)


(1) Scan parquet 
Output [14]: [transaction_id#2419, customer_id#2420, card_id#2421, merchant_id#2422, device_id#2423, timestamp#2424, amount#2425, product_type#2426, card_type#2427, payment_type#2428, customer_region#2429, merchant_region#2430, has_identity#2431, is_fraud#2432]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/fact_transactions.parquet]
ReadSchema: struct<transaction_id:string,customer_id:string,card_id:string,merchant_id:string,device_id:string,timestamp:timestamp,amount:double,product_type:string,card_type:string,payment_type:string,customer_region:string,merchant_regio